# Willingness-to-pay-space mixed logit

Estimate MixL directly in willingness-to-pay (WTP) space with a random time WTP and a fixed cost coefficient.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import numpy as np
import pandas as pd
import torch

from torchdcm import (
    AlternativeScale,
    Beta,
    ChoiceDataset,
    ChoiceLatentEffect,
    ContinuousIndicator,
    CovariateScale,
    CovariateScaledMultinomialLogit,
    CrossNest,
    CrossNestedLogit,
    ErrorComponent,
    ErrorComponentsLogit,
    HybridChoiceModel,
    LatentClassLogit,
    LatentVariable,
    MixedLogit,
    MultinomialLogit,
    Nest,
    NestedLogit,
    OrderedChoiceDataset,
    OrderedLogit,
    OrderedProbit,
    RandomCoefficient,
    ScaledMultinomialLogit,
    UtilitySpec,
    WTPCoefficient,
    WTPMixedLogit,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


def show_result(result):
    print(f"Final log likelihood: {result.loglike:.6f}")
    print(f"AIC: {result.aic:.3f}; BIC: {result.bic:.3f}")
    return pd.DataFrame(
        {
            "estimate": result.values,
            "std. error": result.bse,
            "z": result.zvalues,
            "p-value": result.pvalues,
        },
        index=pd.Index(result.param_names, name="parameter"),
    ).round(6)


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec

frame, data = make_choice_data(seed=15)
spec = UtilitySpec()
spec.utility("TRAIN", Beta("ASC_TRAIN", init=0.0))
spec.utility("SM", Beta("ASC_SM", init=0.0, fixed=True))
spec.utility("CAR", Beta("ASC_CAR", init=0.0))


UtilitySpec(utilities=OrderedDict({'TRAIN': Expression(terms=[Term(parameter=Beta(name='ASC_TRAIN', init=0.0, fixed=False), variable=None, multiplier=1.0)]), 'SM': Expression(terms=[Term(parameter=Beta(name='ASC_SM', init=0.0, fixed=True), variable=None, multiplier=1.0)]), 'CAR': Expression(terms=[Term(parameter=Beta(name='ASC_CAR', init=0.0, fixed=False), variable=None, multiplier=1.0)])}))

In [3]:
model = WTPMixedLogit(
    spec,
    cost=Beta("B_COST", init=-0.10),
    cost_variable="cost",
    wtp_coefficients=[
        WTPCoefficient("WTP_TIME", "time", init=0.10, sigma_init=0.05),
    ],
    n_draws=24,
    seed=15,
    antithetic=True,
    panel=False,
    device=device,
    max_iter=50,
)
result = model.fit(data)
show_result(result)


Final log likelihood: -178.353379
AIC: 366.707; BIC: 382.672


,estimate,std. error,z,p-value
parameter,,,,
ASC_TRAIN,0.262998,0.624260,0.421295,0.673540
ASC_CAR,0.260372,0.653426,0.398472,0.690282
B_COST,-0.068729,0.046289,-1.484777,0.137603
WTP_TIME,0.362647,0.344543,1.052545,0.292550
SIGMA_WTP_TIME,0.479538,0.640342,0.748878,0.453931
